In [4]:
import csv
import time
from collections import Counter

import requests

url = "https://www.wanted.co.kr/api/chaos/navigation/v1/results"
JOB_GROUP_ID = 507   # 원티드 직군 '경영·비즈니스' (wdlist/507)
PAGE_SIZE = 20
MAX_PAGE = 10        # 최대 10페이지(200건)까지만 요청
PM_KEYWORDS = ["PM", "PO", "프로덕트", "Product", "서비스 기획", "서비스기획"]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ko-KR,ko;q=0.9",
    "Referer": "https://www.wanted.co.kr/wdlist/507",
}

rows = []

for page in range(MAX_PAGE):
    params = {
        "job_group_id": JOB_GROUP_ID,
        "country": "kr",
        "job_sort": "job.latest_order",
        "years": -1,
        "locations": "all",
        "limit": PAGE_SIZE,
        "offset": page * PAGE_SIZE,
    }
    res = requests.get(url, params=params, headers=headers, timeout=10)
    res.raise_for_status()
    data = res.json()

    jobs = data.get("data", [])
    if not jobs:
        break

    if page == 0:
        print("공고 한 건에 들어 있는 항목:", list(jobs[0].keys()))
        print()

    for job in jobs:
        position = job.get("position", "")
        if not any(word.lower() in position.lower() for word in PM_KEYWORDS):
            continue

        company = job.get("company") or {}
        address = job.get("address") or {}
        rows.append({
            "공고ID": job.get("id"),
            "포지션": position,
            "회사": company.get("name", ""),
            "산업": company.get("industry_name", ""),
            "지역": address.get("location", ""),
            "구": address.get("district", ""),
            "최소경력": job.get("annual_from", ""),
            "최대경력": job.get("annual_to", ""),
            "마감일": job.get("due_time") or "상시",
            "링크": f"https://www.wanted.co.kr/wd/{job.get('id')}",
        })

    print(f"{page + 1}페이지까지 확인했습니다. PM 관련 공고 누적 {len(rows)}건")

    if not (data.get("links") or {}).get("next"):
        break

    time.sleep(1)

with open("wanted_pm_jobs.csv", "w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()) if rows else ["공고ID"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nwanted_pm_jobs.csv에 {len(rows)}건을 저장했습니다.\n")

# ---------- 간단한 분석 ----------
if rows:
    def year_group(value):
        if value == "" or value is None:
            return "정보 없음"
        value = int(value)
        if value == 0:
            return "신입 가능"
        if value <= 2:
            return "1~2년"
        if value <= 5:
            return "3~5년"
        return "6년 이상"

    print("[최소 경력 분포]")
    for name, count in Counter(year_group(r["최소경력"]) for r in rows).most_common():
        print(f"  {name}: {count}건")

    print("\n[지역 상위 5개]")
    for name, count in Counter(r["지역"] or "정보 없음" for r in rows).most_common(5):
        print(f"  {name}: {count}건")

    print("\n[산업 상위 5개]")
    for name, count in Counter(r["산업"] or "정보 없음" for r in rows).most_common(5):
        print(f"  {name}: {count}건")

    print("\n[마감 형태]")
    always = sum(1 for r in rows if r["마감일"] == "상시")
    print(f"  상시채용: {always}건 / 마감일 있음: {len(rows) - always}건")

공고 한 건에 들어 있는 항목: ['id', 'reward_total', 'is_bookmark', 'company', 'title_img', 'address', 'position', 'category_tag', 'attraction_tags', 'skill_tags', 'user_oriented_tags', 'annual_from', 'annual_to', 'is_newbie', 'reward', 'employment_type', 'additional_apply_type', 'preferred_languages', 'visa_information', 'is_outlink']

1페이지까지 확인했습니다. PM 관련 공고 누적 7건
2페이지까지 확인했습니다. PM 관련 공고 누적 10건
3페이지까지 확인했습니다. PM 관련 공고 누적 13건
4페이지까지 확인했습니다. PM 관련 공고 누적 17건
5페이지까지 확인했습니다. PM 관련 공고 누적 22건
6페이지까지 확인했습니다. PM 관련 공고 누적 25건
7페이지까지 확인했습니다. PM 관련 공고 누적 31건
8페이지까지 확인했습니다. PM 관련 공고 누적 35건
9페이지까지 확인했습니다. PM 관련 공고 누적 41건
10페이지까지 확인했습니다. PM 관련 공고 누적 47건

wanted_pm_jobs.csv에 47건을 저장했습니다.

[최소 경력 분포]
  3~5년: 30건
  6년 이상: 8건
  1~2년: 6건
  신입 가능: 3건

[지역 상위 5개]
  서울: 44건
  경기: 2건
  인천: 1건

[산업 상위 5개]
  정보 없음: 47건

[마감 형태]
  상시채용: 47건 / 마감일 있음: 0건


In [5]:
import csv
import time
from collections import Counter

import requests

url = "https://www.wanted.co.kr/api/chaos/navigation/v1/results"
JOB_GROUP_ID = 507   # 원티드 직군 '경영·비즈니스' (wdlist/507)
PAGE_SIZE = 20
MAX_PAGE = 10        # 최대 10페이지(200건)까지만 요청
PM_KEYWORDS = ["PM", "PO", "프로덕트", "Product", "서비스 기획", "서비스기획"]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ko-KR,ko;q=0.9",
    "Referer": "https://www.wanted.co.kr/wdlist/507",
}

rows = []

for page in range(MAX_PAGE):
    params = {
        "job_group_id": JOB_GROUP_ID,
        "country": "kr",
        "job_sort": "job.latest_order",
        "years": -1,
        "locations": "all",
        "limit": PAGE_SIZE,
        "offset": page * PAGE_SIZE,
    }
    res = requests.get(url, params=params, headers=headers, timeout=10)
    res.raise_for_status()
    data = res.json()

    jobs = data.get("data", [])
    if not jobs:
        break

    if page == 0:
        print("공고 한 건에 들어 있는 항목:", list(jobs[0].keys()))
        print()

    for job in jobs:
        position = job.get("position", "")
        if not any(word.lower() in position.lower() for word in PM_KEYWORDS):
            continue

        company = job.get("company") or {}
        address = job.get("address") or {}
        rows.append({
            "공고ID": job.get("id"),
            "포지션": position,
            "회사": company.get("name", ""),
            "지역": address.get("location", ""),
            "구": address.get("district", ""),
            "최소경력": job.get("annual_from", ""),
            "최대경력": job.get("annual_to", ""),
            "링크": f"https://www.wanted.co.kr/wd/{job.get('id')}",
        })

    print(f"{page + 1}페이지까지 확인했습니다. PM 관련 공고 누적 {len(rows)}건")

    if not (data.get("links") or {}).get("next"):
        break

    time.sleep(1)

with open("wanted_pm_jobs.csv", "w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()) if rows else ["공고ID"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nwanted_pm_jobs.csv에 {len(rows)}건을 저장했습니다.\n")

# ---------- 간단한 분석 ----------
if rows:
    def year_group(value):
        if value == "" or value is None:
            return "정보 없음"
        value = int(value)
        if value == 0:
            return "신입 가능"
        if value <= 2:
            return "1~2년"
        if value <= 5:
            return "3~5년"
        return "6년 이상"

    print("[최소 경력 분포]")
    for name, count in Counter(year_group(r["최소경력"]) for r in rows).most_common():
        print(f"  {name}: {count}건")

    print("\n[지역 상위 5개]")
    for name, count in Counter(r["지역"] or "정보 없음" for r in rows).most_common(5):
        print(f"  {name}: {count}건")

    print("\n[서울 공고의 구별 상위 5개]")
    seoul = [r for r in rows if r["지역"] == "서울"]
    for name, count in Counter(r["구"] or "정보 없음" for r in seoul).most_common(5):
        print(f"  {name}: {count}건")

공고 한 건에 들어 있는 항목: ['id', 'reward_total', 'is_bookmark', 'company', 'title_img', 'address', 'position', 'category_tag', 'attraction_tags', 'skill_tags', 'user_oriented_tags', 'annual_from', 'annual_to', 'is_newbie', 'reward', 'employment_type', 'additional_apply_type', 'preferred_languages', 'visa_information', 'is_outlink']

1페이지까지 확인했습니다. PM 관련 공고 누적 7건
2페이지까지 확인했습니다. PM 관련 공고 누적 10건
3페이지까지 확인했습니다. PM 관련 공고 누적 13건
4페이지까지 확인했습니다. PM 관련 공고 누적 17건
5페이지까지 확인했습니다. PM 관련 공고 누적 22건
6페이지까지 확인했습니다. PM 관련 공고 누적 24건
7페이지까지 확인했습니다. PM 관련 공고 누적 30건
8페이지까지 확인했습니다. PM 관련 공고 누적 35건
9페이지까지 확인했습니다. PM 관련 공고 누적 41건
10페이지까지 확인했습니다. PM 관련 공고 누적 47건

wanted_pm_jobs.csv에 47건을 저장했습니다.

[최소 경력 분포]
  3~5년: 30건
  6년 이상: 8건
  1~2년: 6건
  신입 가능: 3건

[지역 상위 5개]
  서울: 44건
  경기: 2건
  인천: 1건

[서울 공고의 구별 상위 5개]
  강남구: 20건
  서초구: 7건
  종로구: 4건
  성동구: 3건
  마포구: 3건
